In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.7.2 which is incompatible.


In [3]:
import os, sys, re, time, torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error: Path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("✓ Environment Ready")

✓ Environment Ready


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# فشرده سازی 4 بیتی حذف شد. مدل به صورت خام و بسیار سریع روی T4 لود می شود
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
print(f"✓ Model loaded in FAST MODE: {MODEL_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

✓ Model loaded in FAST MODE: Qwen/Qwen2.5-Math-1.5B-Instruct


In [10]:
import re
import time
import torch
from sympy import symbols, simplify, N

def solve_latex_expression(question_text: str) -> str | None:
    try:
        from latex2sympy2 import latex2sympy
        latex_blocks = re.findall(r'\$([^$]+)\$', question_text)
        if not latex_blocks: return None
        main_latex = max(latex_blocks, key=len)
        expr = latex2sympy(main_latex)
        result = float(N(simplify(expr)))
        return str(int(result)) if result == int(result) else str(round(result, 6))
    except Exception:
        return None

def match_result_to_option(computed: str, options: list) -> int | None:
    try: computed_val = float(computed)
    except ValueError: return None
    for i, opt in enumerate(options):
        opt_clean = re.sub(r'\\[a-zA-Z]+\{?|\}', '', opt.text).strip()
        for n in re.findall(r'-?\d+\.?\d*', opt_clean):
            try:
                if abs(float(n) - computed_val) < 1e-4: return i
            except ValueError: continue
    return None

def classify_math_type(text: str) -> str:
    text = text.lower()
    if '$' in text and re.search(r'\$[^$]+\$', text): return 'latex_algebra'
    if any(k in text for k in ['standard deviation', 'probability', 'z-score', 't-test', 'mean']): return 'stats_probability'
    if any(k in text for k in ['derivative', 'integral', 'velocity', 'rate']): return 'calculus'
    return 'conceptual'

def choose_answer_math_fixed(question, tokenizer, model) -> tuple:
    options_text = "\n".join(f"{chr(65+i)}) {opt.text}" for i, opt in enumerate(question.options))
    math_type = classify_math_type(question.text)

    # 1. SymPy برای حل سریع
    if math_type == 'latex_algebra':
        computed = solve_latex_expression(question.text)
        if computed:
            matched_idx = match_result_to_option(computed, question.options)
            if matched_idx is not None:
                return question.options[matched_idx].id, chr(65 + matched_idx), f"sympy→{computed}"

    # 2. پرامپت با محدودیت توکن بالاتر برای جلوگیری از قطع شدن استدلال
    messages = [
        {"role": "system", "content": "You are a math solver. Reason step-by-step but BE EXTREMELY BRIEF. DO NOT write long paragraphs. Conclude with your final option letter inside a box, like \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
        {"role": "user", "content": f"{question.text}\n\nOptions:\n{options_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    THINK_TOKENS = 480
    WRAPUP_TOKENS = 20
    gen_start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=THINK_TOKENS,
            max_time=27.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    gen_tokens = outputs[0].shape[0] - input_len
    think_elapsed = time.time() - gen_start
    almost_out_of_time = think_elapsed >= 26.5
    almost_out_of_tokens = gen_tokens >= THINK_TOKENS
    wrapup_used = False
    wrapup_raw = ''

    if almost_out_of_time or almost_out_of_tokens:
        wrap_messages = messages + [
            {"role": "assistant", "content": raw},
            {"role": "user", "content": "Time or token budget is almost finished. Stop reasoning and submit the nearest guess now. Reply only with one final option: \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."}
        ]
        wrap_prompt = tokenizer.apply_chat_template(wrap_messages, tokenize=False, add_generation_prompt=True)
        wrap_inputs = tokenizer(wrap_prompt, return_tensors='pt').to(model.device)
        wrap_input_len = wrap_inputs['input_ids'].shape[1]
        remaining_time = 28.5 - (time.time() - gen_start)

        if remaining_time > 0.5:
            with torch.no_grad():
                wrap_outputs = model.generate(
                    **wrap_inputs,
                    max_new_tokens=WRAPUP_TOKENS,
                    max_time=remaining_time,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            wrapup_raw = tokenizer.decode(wrap_outputs[0][wrap_input_len:], skip_special_tokens=True)
            raw += "\n" + wrapup_raw
            wrapup_used = True

    total_elapsed = time.time() - gen_start
    wrapup_reason = []
    if almost_out_of_time: wrapup_reason.append('time')
    if almost_out_of_tokens: wrapup_reason.append('tokens')
    print(f"[Monitor] think_tokens={gen_tokens}/{THINK_TOKENS} | think_time={think_elapsed:.2f}s | total_time={total_elapsed:.2f}s | wrapup={wrapup_used} | reason={','.join(wrapup_reason) or 'none'}")
    if wrapup_used:
        print(f"[Wrapup Raw] {wrapup_raw.strip()}")
    print(f"[Model Tail]\n{raw.strip()[-700:]}")
    print(f"\n[Model Log]\n{raw.strip()}\n[End Log]")

    # 3. سیستم استخراج

    # اولویت اول: پیدا کردن \boxed{X}
    boxed_matches = re.findall(r'\\boxed\{([A-D])\}', raw, re.IGNORECASE)
    if boxed_matches:
        letter = boxed_matches[-1].upper()
        idx = ['A', 'B', 'C', 'D'].index(letter)
        print(f"[Extract] rule=boxed | letter={letter} | all_boxed={boxed_matches}")
        return question.options[idx].id, letter, f"boxed_{math_type}"

    # اولویت دوم: پیدا کردن کلماتی مثل Option X یا Answer: X
    match = re.search(r'(?:FINAL ANSWER|option|answer is|choice)\s*[:]?\s*([A-D])\b', raw, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        idx = ['A', 'B', 'C', 'D'].index(letter)
        print(f"[Extract] rule=text_match | letter={letter} | matched='{match.group(0)}'")
        return question.options[idx].id, letter, f"text_match_{math_type}"

    # اولویت سوم (فال‌بک نهایی): آخرین حرف انگلیسی مجزا در متن
    matches = re.findall(r'\b([A-D])\b', raw.strip().upper())
    letter = matches[-1] if matches else 'C'
    try:
        idx = ['A', 'B', 'C', 'D'].index(letter)
    except ValueError:
        idx = 2
        letter = 'C'

    print(f"[Extract] rule=fallback | letter={letter} | candidates={matches[-10:]}")
    return question.options[idx].id, letter, f"fallback_{math_type}"

In [12]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

client = MillionaireClient('http://131.175.15.22:51111/')
user   = client.login('gary', '13790229')
print(f"✓ Logged in as: {user.username}")

competitions = client.competitions.list_all()
# فرض بر این است که ایندکس 3 مربوط به Maths است. اگر تغییر کرده، عدد 3 را اصلاح کن
COMPETITION_ID = competitions[3].id

game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None: break

    print(f"\n{'='*60}\nLevel: {game.current_level}\nQ: {question.text}")
    for i, opt in enumerate(question.options):
        print(f"  {chr(65+i)}) {opt.text}")

    t0 = time.time()
    option_id, letter, method = choose_answer_math_fixed(question, tokenizer, model)
    t1 = time.time()

    print(f"→ Predicted: {letter} | Method: {method} | Time taken: {t1-t0:.2f}s")

    try:
        result = game.answer(option_id)
        print(f"Correct: {result.correct} | Earned: {result.earned_amount}")
    except TimeoutError:
        print("✗ Timed out (Generation took >30s)")
        break
    except RateLimitError:
        print("⚠ Rate limited — waiting 5s")
        time.sleep(5)
        result = game.answer(option_id)
        print(f"Correct: {result.correct} | Earned: {result.earned_amount}")

    if result.game_over: break
    time.sleep(1)

print(f"\n{'='*60}\n✓ Game over. Final score: {game.earned_amount}")

✓ Logged in as: gary

Level: 1
Q: If the parabola $y_1 = x^2 + 2x + 7$ and the line $y_2 = 6x + b$ intersect at only one point, what is the value of $b$?
  A) 4
  B) 3
  C) 7
  D) 12
[Monitor] think_tokens=468/480 | think_time=19.53s | total_time=19.53s | wrapup=False | reason=none
[Model Tail]
= 1 \), \( b = -4 \), and \( c = 7 - b \). For the quadratic equation to have exactly one solution, the discriminant must be zero. The discriminant \(\Delta\) of a quadratic equation \( ax^2 + bx + c = 0 \) is given by:
\[ \Delta = b^2 - 4ac. \]

Substitute the values of \( a \), \( b \), and \( c \) into the discriminant formula:
\[ \Delta = (-4)^2 - 4(1)(7 - b), \]
\[ \Delta = 16 - 4(7 - b), \]
\[ \Delta = 16 - 28 + 4b, \]
\[ \Delta = 4b - 12. \]

Set the discriminant equal to zero for the quadratic equation to have exactly one solution:
\[ 4b - 12 = 0. \]

Solve for \( b \):
\[ 4b = 12, \]
\[ b = 3. \]

Therefore, the value of \( b \) is \(\boxed{3}\). The correct choice is \(\boxed{B}\).

[M